# 001 · Paths with pathlib

> **Standard Library › Files & Paths** · Python 3.14

`pathlib` gives you an object-oriented `Path` type that both *describes* a path (name, extension, parent…) and *acts* on the file system (exists, mkdir, read, glob…). It is the modern, preferred way to work with paths in Python.

**In this notebook you will learn:**

- how `Path` compares to `System.IO.Path`, `FileInfo` and `DirectoryInfo`
- joining paths with the `/` operator and taking them apart (`name`, `stem`, `suffix`, `parent`, `parts`)
- checking, creating, listing and searching files and directories (`glob`, `rglob`, `walk`)
- reading/writing small files in one call, resolving and relativising paths
- pure paths for Windows/POSIX manipulation, and the 3.14 additions `copy`, `move` and `info`

## .NET mental model

| .NET | Python |
|---|---|
| `System.IO.Path` (static string helpers) | `pathlib.PurePath` / `Path` methods and properties (`os.path` is the legacy string API) |
| `FileInfo` / `DirectoryInfo` | a single type: `pathlib.Path` (file *or* directory) |
| `Path.Combine(a, b)` | `a / b` |
| `File.ReadAllText` / `WriteAllText` | `p.read_text()` / `p.write_text()` |
| `Directory.EnumerateFiles(dir, "*.cs", SearchOption.AllDirectories)` | `p.rglob("*.cs")` |

The key mental shift: in .NET a path is usually a `string` that you pass to static helpers. In Python a path is a **value object** — immutable, hashable, comparable — with the helpers as methods. Most standard library functions (`open`, `shutil`, `os`, …) accept `Path` objects directly.

## Creating and joining paths

```csharp
// C#
var config = Path.Combine("app", "config", "settings.json");
var info   = new FileInfo(config);
```

Python overloads the `/` operator for joining. `Path(...)` on its own builds a path for the current OS (`PosixPath` on macOS/Linux, `WindowsPath` on Windows).

In [ ]:
from pathlib import Path

config = Path("app") / "config" / "settings.json"
print(config)              # app/config/settings.json  (backslashes on Windows)
print(type(config).__name__)
print(Path("app", "config", "settings.json") == config)   # constructor also joins

⚠️ **Gotcha:** joining an **absolute** path discards everything before it — exactly like `Path.Combine("app", "/etc")` in .NET.

In [ ]:
print(Path("app") / "/etc" / "hosts")   # /etc/hosts — "app" is gone

## Taking a path apart

| `System.IO.Path` | `pathlib.Path` |
|---|---|
| `GetFileName(p)` | `p.name` |
| `GetFileNameWithoutExtension(p)` | `p.stem` |
| `GetExtension(p)` | `p.suffix` (and `p.suffixes` for `.tar.gz`) |
| `GetDirectoryName(p)` | `p.parent` |
| `ChangeExtension(p, ".bak")` | `p.with_suffix(".bak")` |

In [ ]:
p = Path("data/reports/2026/summary.final.csv")

print(p.name)        # summary.final.csv
print(p.stem)        # summary.final
print(p.suffix)      # .csv
print(p.suffixes)    # ['.final', '.csv']
print(p.parent)      # data/reports/2026
print(p.parts)       # ('data', 'reports', '2026', 'summary.final.csv')
print([str(a) for a in p.parents][:2])   # ancestors, nearest first

Paths are immutable — the `with_*` methods return **new** paths:

In [ ]:
print(p.with_suffix(".json"))         # data/reports/2026/summary.final.json
print(p.with_name("index.html"))      # data/reports/2026/index.html
print(p.with_stem("draft"))           # data/reports/2026/draft.csv
print(p.parent.parent / "2025")       # navigating up, then down

## A sandbox to play in

From here on we work with real files, always inside a temporary directory that is deleted automatically *(see Standard Library › Files & Paths › 004 · Temporary Files and Directories)*. The `show` helper prints paths **relative** to the sandbox so the output is the same on every machine.

In [ ]:
import tempfile

tmp = tempfile.TemporaryDirectory()
root = Path(tmp.name)

def show(path: Path) -> str:
    return path.relative_to(root).as_posix()

print(show(root / "a" / "b.txt"))

## Checking, creating and writing

```csharp
// C#
Directory.CreateDirectory("logs/2026/10");     // creates parents, no error if it exists
File.WriteAllText("logs/2026/10/app.log", "started");
bool ok = File.Exists("logs/2026/10/app.log");
```

`mkdir()` needs two flags to behave like `Directory.CreateDirectory`: `parents=True` (create missing ancestors) and `exist_ok=True` (no error if it already exists).

In [ ]:
logs = root / "logs" / "2026" / "10"
logs.mkdir(parents=True, exist_ok=True)

log_file = logs / "app.log"
log_file.write_text("started\n", encoding="utf-8")   # returns number of chars written

print(log_file.exists(), log_file.is_file(), log_file.is_dir())   # True True False
print(logs.is_dir())                                             # True
print((root / "missing.txt").exists())                           # False

⚠️ **Gotcha:** without the flags, `mkdir` raises — unlike .NET's forgiving `CreateDirectory`.

In [ ]:
try:
    (root / "x" / "y").mkdir()             # parent "x" does not exist
except FileNotFoundError as e:
    print(f"{type(e).__name__}: {e.strerror}")

try:
    logs.mkdir()                            # already exists
except FileExistsError as e:
    print(f"{type(e).__name__}: {e.strerror}")

## Reading and writing whole files

`read_text` / `write_text` / `read_bytes` / `write_bytes` are the one-liners — the equivalents of `File.ReadAllText`, `File.WriteAllText`, `File.ReadAllBytes`, `File.WriteAllBytes`. For streaming or appending use `open()` *(see Standard Library › Files & Paths › 002 · Reading and Writing Files)*.

**Tip:** always pass `encoding="utf-8"`. Unlike .NET (which defaults to UTF-8), Python's default text encoding can depend on the OS locale.

In [ ]:
notes = root / "notes.md"
notes.write_text("# Notes\n- café ☕\n", encoding="utf-8")

print(notes.read_text(encoding="utf-8"))
print(notes.read_bytes()[:12])          # raw bytes — é is two bytes in UTF-8
print(notes.stat().st_size, "bytes")    # FileInfo.Length

## Listing and searching

| .NET | Python |
|---|---|
| `Directory.EnumerateFileSystemEntries(dir)` | `dir.iterdir()` |
| `Directory.EnumerateFiles(dir, "*.csv")` | `dir.glob("*.csv")` |
| `Directory.EnumerateFiles(dir, "*.csv", SearchOption.AllDirectories)` | `dir.rglob("*.csv")` or `dir.glob("**/*.csv")` |

All of these are **lazy** generators (like `Enumerate*`, not `GetFiles`). The order is whatever the OS returns, so sort when you need stable output.

In [ ]:
for name in ["src/main.py", "src/util/io.py", "src/util/__init__.py", "tests/test_main.py", "README.md"]:
    f = root / "proj" / name
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text("", encoding="utf-8")

proj = root / "proj"
print(sorted(p.name for p in proj.iterdir()))              # direct children only
print(sorted(show(p) for p in proj.glob("src/*.py")))       # one level
print(sorted(show(p) for p in proj.rglob("*.py")))          # recursive

Glob patterns support `*`, `?`, `[abc]` and `**` (any number of directories). Filter files vs directories with `is_file()`:

In [ ]:
print(sorted(p.name for p in proj.rglob("*") if p.is_dir()))     # ['src', 'tests', 'util']
print(sorted(p.name for p in proj.glob("**/test_*.py")))         # ['test_main.py']

### Walking a tree (3.12+)

`Path.walk()` (added in 3.12) yields `(dirpath, dirnames, filenames)` for every directory, like the classic `os.walk` but with `Path` objects. You can prune the walk by modifying `dirnames` in place.

In [ ]:
for dirpath, dirnames, filenames in proj.walk():
    dirnames.sort()                   # in-place: controls order AND which dirs are visited
    if "tests" in dirnames:
        dirnames.remove("tests")      # prune: don't descend into tests/
    print(f"{show(dirpath):<18} dirs={dirnames} files={sorted(filenames)}")

## Absolute, resolved and relative paths

| .NET | Python |
|---|---|
| `Path.GetFullPath(p)` | `p.resolve()` (also follows symlinks, removes `..`) or `p.absolute()` |
| `Path.GetRelativePath(baseDir, p)` | `p.relative_to(base_dir)` |
| `Path.IsPathRooted(p)` | `p.is_absolute()` |
| `Environment.CurrentDirectory` | `Path.cwd()` |
| `Environment.GetFolderPath(SpecialFolder.UserProfile)` | `Path.home()` |

In [ ]:
messy = root / "proj" / "src" / ".." / "tests" / "test_main.py"
clean = messy.resolve()

print(".." in messy.parts, ".." in clean.parts)     # True False
print(clean.relative_to(root.resolve()))            # proj/tests/test_main.py
print(Path("relative/file.txt").is_absolute())      # False
print(Path.cwd().is_absolute(), Path.home().is_absolute())   # True True

⚠️ **Gotcha:** `relative_to` is purely lexical and raises `ValueError` if the path is not under the base. Since 3.12 you can pass `walk_up=True` to get `..` segments, like `Path.GetRelativePath` does.

In [ ]:
a = Path("/srv/app/logs")
b = Path("/srv/data")
try:
    b.relative_to(a)
except ValueError as e:
    print(f"{type(e).__name__}: {e}")
print(b.relative_to(a, walk_up=True))   # ../../data

## Pure paths: Windows vs POSIX

`PurePath` objects do string manipulation only — they never touch the disk. `PureWindowsPath` and `PurePosixPath` let you handle paths **for another OS**, e.g. parsing Windows paths from a config file while running on Linux.

In [ ]:
from pathlib import PurePosixPath, PureWindowsPath

win = PureWindowsPath(r"C:\Users\alice\Documents\Report.DOCX")
print(win.drive, win.name, win.suffix)          # C: Report.DOCX .DOCX
print(win.parts)
print(win.as_posix())                           # forward slashes
print(win == PureWindowsPath(r"c:\users\ALICE\documents\report.docx"))   # True — case-insensitive

posix = PurePosixPath("/var/log") / "app.log"
print(posix, posix == PurePosixPath("/VAR/log/app.log"))   # case-sensitive -> False

**Tip:** use `p.as_posix()` when you need a stable string with `/` separators (URLs, logs, test output) and `str(p)` when handing the path to an OS-specific API.

## New in 3.14: `copy`, `move` and `info`

Python 3.14 added file-system operations that previously required `shutil` *(see Standard Library › Files & Paths › 003 · File System Operations with os and shutil)*:

- `p.copy(target)` — copy a file **or a whole directory tree**; returns the new `Path`
- `p.copy_into(target_dir)` — copy into an existing directory, keeping the name
- `p.move(target)` / `p.move_into(target_dir)` — move/rename (works across file systems)
- `p.info` — a cached `PathInfo` with `exists()`, `is_file()`, `is_dir()`, `is_symlink()`

In [ ]:
backup_dir = root / "backup"
backup_dir.mkdir()

copied = notes.copy(backup_dir / "notes-copy.md")    # File.Copy
into = notes.copy_into(backup_dir)                   # keeps the name
tree = proj.copy(root / "proj-copy")                 # whole tree, like a recursive copy
moved = copied.move(root / "archived.md")            # File.Move

print(show(copied), copied.exists())     # source of the move is gone
print(show(into), show(moved), show(tree))
print(sorted(show(p) for p in backup_dir.iterdir()))

⚠️ **Gotcha:** `copy` does **not** create missing parent directories, and copying a directory onto an existing one raises `FileExistsError`.

In [ ]:
try:
    notes.copy(root / "no" / "such" / "dir" / "notes.md")
except FileNotFoundError as e:
    print(f"{type(e).__name__}: {e.strerror}")

try:
    proj.copy(root / "proj-copy")
except FileExistsError as e:
    print(f"{type(e).__name__}: {e.strerror}")

`Path.info` is handy when iterating: paths from `iterdir()` may already know their type from the directory scan, which saves extra system calls.

In [ ]:
for entry in sorted(proj.iterdir()):
    kind = "dir " if entry.info.is_dir() else "file"
    print(kind, entry.name)

## Cleaning up

`TemporaryDirectory.cleanup()` removes the sandbox and everything in it (normally you'd use it as a `with` block).

In [ ]:
tmp.cleanup()
print(root.exists())   # False

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Join | `Path.Combine(a, b)` | `Path(a) / b` |
| File name / stem / extension | `GetFileName` / `GetFileNameWithoutExtension` / `GetExtension` | `.name` / `.stem` / `.suffix` |
| Parent directory | `GetDirectoryName` / `DirectoryInfo.Parent` | `.parent` |
| Change extension | `Path.ChangeExtension(p, ".bak")` | `p.with_suffix(".bak")` |
| Exists? | `File.Exists` / `Directory.Exists` | `p.exists()`, `p.is_file()`, `p.is_dir()` |
| Create directory tree | `Directory.CreateDirectory(p)` | `p.mkdir(parents=True, exist_ok=True)` |
| Read/write whole file | `File.ReadAllText` / `WriteAllText` | `p.read_text(encoding=...)` / `p.write_text(...)` |
| Enumerate recursively | `EnumerateFiles(d, "*.py", SearchOption.AllDirectories)` | `d.rglob("*.py")` |
| Walk a tree | — | `d.walk()` (3.12+) |
| Full path | `Path.GetFullPath(p)` | `p.resolve()` |
| Relative path | `Path.GetRelativePath(b, p)` | `p.relative_to(b)` |
| Copy / move | `File.Copy` / `File.Move` | `p.copy(t)` / `p.move(t)` (3.14+) |

## Exercises

1. Given `Path("photos/2026/holiday/IMG_0042.JPG")`, print the name in lower case, a version of the path with a `.webp` extension, and the year folder (`2026`) using `parts` or `parents`.
2. In a temporary directory, create `a/one.txt`, `a/b/two.txt`, `a/b/c/three.log`. Then print a sorted list of all `.txt` files relative to the temp dir (POSIX style).
3. Write a function `total_size(folder: Path) -> int` that returns the total size in bytes of all files under `folder` (recursively).

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import tempfile
from pathlib import Path

# 1 — properties and with_suffix
photo = Path("photos/2026/holiday/IMG_0042.JPG")
print(photo.name.lower())            # img_0042.jpg
print(photo.with_suffix(".webp"))    # photos/2026/holiday/IMG_0042.webp
print(photo.parts[1], photo.parents[1].name)   # 2026 2026

# 2 — create files, then rglob
with tempfile.TemporaryDirectory() as d:
    base = Path(d)
    for rel in ["a/one.txt", "a/b/two.txt", "a/b/c/three.log"]:
        f = base / rel
        f.parent.mkdir(parents=True, exist_ok=True)
        f.write_text("hello", encoding="utf-8")
    print(sorted(p.relative_to(base).as_posix() for p in base.rglob("*.txt")))

# 3 — sum stat().st_size over every file
def total_size(folder: Path) -> int:
    return sum(p.stat().st_size for p in folder.rglob("*") if p.is_file())

with tempfile.TemporaryDirectory() as d:
    base = Path(d)
    (base / "x.bin").write_bytes(b"\0" * 100)
    (base / "sub").mkdir()
    (base / "sub" / "y.bin").write_bytes(b"\0" * 23)
    print(total_size(base))          # 123
```

</details>